In [2]:
import pandas as pd
import matplotlib as plt

In [3]:
df = pd.read_csv('zeroshot_qwen3_30b.csv')
df.head(3)

,Problem,Level,Hardware,Device,Compiled,Correctness,Ref_PyTorch_Runtime_ms,Ref_PyTorch_Compiled_Runtime_ms,Triton_Runtime_ms,Speedup,Speedup_vs_Compiled,Status,Error_Message,Timestamp
0,100_HingeLoss,level1,NVIDIA A100-SXM4-80GB,0.0,True,False,-1.0,-1.0,-1.0,-1.0,-1.0,SUCCESS,NaN,2025-11-09 22:16:03
1,10_3D_tensor_matrix_multiplication,level1,NVIDIA A100-SXM4-80GB,0.0,True,False,-1.0,-1.0,-1.0,-1.0,-1.0,SUCCESS,NaN,2025-11-09 22:16:06
2,11_4D_tensor_matrix_multiplication,level1,NVIDIA A100-SXM4-80GB,0.0,True,False,-1.0,-1.0,-1.0,-1.0,-1.0,SUCCESS,NaN,2025-11-09 22:16:11


In [4]:
df_L1 = df[df["Level"] == "level1"]
df_L2 = df[df["Level"] == "level2"]
df_L3 = df[df["Level"] == "level3"]

In [6]:
print(df_L1.shape)
print(df_L2.shape)
(print(df_L3.shape))

(100, 14)
(100, 14)
(50, 14)


In [10]:
print(df_L1["Compiled"].value_counts())
print(df_L1["Correctness"].value_counts())

Compiled
True     72
False    21
Name: count, dtype: int64
Correctness
False    86
True      7
Name: count, dtype: int64


In [11]:
print(df_L2["Compiled"].value_counts())
print(df_L2["Correctness"].value_counts())

Compiled
True     76
False    21
Name: count, dtype: int64
Correctness
False    96
True      1
Name: count, dtype: int64


In [12]:
print(df_L3["Compiled"].value_counts())
print(df_L3["Correctness"].value_counts())

Compiled
True     27
False     8
Name: count, dtype: int64
Correctness
False    35
Name: count, dtype: int64


In [ ]:
tmp = df_L1[df_L1["Correctness"] == True]
L1_speedup = tmp.Speedup.values
L1_compiled_speedup = tmp.Speedup_vs_Compiled.values

In [22]:
tmp = df_L2[df_L2["Correctness"] == True]
L2_speedup = tmp.Speedup.values
L2_compiled_speedup = tmp.Speedup_vs_Compiled.values

In [29]:
print(L1_speedup.mean())
print(L1_compiled_speedup.mean())

print(L2_speedup.mean())
print(L2_compiled_speedup.mean())

10.395714285714288
10.675714285714287
0.03
0.03


# Model Comparison: GPT-5, Gemini, and Search GPT-5

In [20]:
import os
import numpy as np
import pandas as pd

def analyze_model_csv(csv_path, model_name):
    """
    Analyze a model's CSV file and return statistics for each level.
    
    Args:
        csv_path: Path to the CSV file
        model_name: Name of the model for display
    
    Returns:
        DataFrame with statistics per level
    """
    if not os.path.exists(csv_path):
        print(f"File not found: {csv_path}")
        return None
    
    df = pd.read_csv(csv_path)
    
    results = []
    
    for level in ['level1', 'level2', 'level3']:
        df_level = df[df['Level'] == level]
        
        if len(df_level) == 0:
            continue
        
        # Count compiled and correctness
        total = len(df_level)
        compiled_count = df_level['Compiled'].sum()
        correct_count = df_level['Correctness'].sum()
        
        # Calculate speedup statistics for correct results only
        correct_df = df_level[df_level['Correctness'] == True]
        
        # Calculate fast_1 and fast_2 metrics for uncompiled baseline
        # fast_1: correct and faster than PyTorch baseline (speedup > 1.0)
        fast_1_count = len(df_level[(df_level['Correctness'] == True) & (df_level['Speedup'] > 1.0)])
        # fast_2: correct and at least 2x faster than PyTorch baseline (speedup >= 2.0)
        fast_2_count = len(df_level[(df_level['Correctness'] == True) & (df_level['Speedup'] >= 2.0)])
        
        # Calculate fast_1 and fast_2 metrics for compiled baseline
        fast_1_comp_count = len(df_level[(df_level['Correctness'] == True) & (df_level['Speedup_vs_Compiled'] > 1.0)])
        fast_2_comp_count = len(df_level[(df_level['Correctness'] == True) & (df_level['Speedup_vs_Compiled'] >= 2.0)])
        
        if len(correct_df) > 0:
            # Speedup (vs uncompiled PyTorch)
            speedup_vals = correct_df['Speedup'].values
            mean_speedup = np.mean(speedup_vals)
            std_speedup = np.std(speedup_vals)
            
            # Speedup vs compiled PyTorch
            speedup_comp_vals = correct_df['Speedup_vs_Compiled'].values
            mean_speedup_compiled = np.mean(speedup_comp_vals)
            std_speedup_compiled = np.std(speedup_comp_vals)
        else:
            mean_speedup = std_speedup = 0.0
            mean_speedup_compiled = std_speedup_compiled = 0.0
        
        results.append({
            'Model': model_name,
            'Level': level.replace('level', 'L'),
            'Total': total,
            'Compiled': f"{compiled_count}/{total} ({100*compiled_count/total:.1f}%)",
            'Correct': f"{correct_count}/{total} ({100*correct_count/total:.1f}%)",
            'Fast1/Fast2': f"{fast_1_count}/{fast_2_count}",
            'Speedup': f"{mean_speedup:.2f}x ± {std_speedup:.2f}",
            'Fast1/Fast2_Comp': f"{fast_1_comp_count}/{fast_2_comp_count}",
            'SpeedupComp': f"{mean_speedup_compiled:.2f}x ± {std_speedup_compiled:.2f}"
        })
    
    return pd.DataFrame(results)

def compare_selected_models(model_list):
    """
    Compare selected model CSV files and return a comprehensive table.
    
    Args:
        model_list: List of tuples (csv_file, model_name)
    """
    all_results = []
    
    for csv_file, model_name in model_list:
        result = analyze_model_csv(csv_file, model_name)
        if result is not None:
            all_results.append(result)
    
    if all_results:
        combined_df = pd.concat(all_results, ignore_index=True)
        return combined_df
    else:
        return None


## Overall Results


In [22]:
# Compare GPT-5, Gemini, and Search GPT-5 (No level breakdown)
models_to_compare = [
    ('zeroshot_gpt-5.csv', 'GPT-5'),
    ('zeroshot_gemini-2.5-pro.csv', 'Gemini-2.5-Pro'),
    ('search_openai-gpt-5.csv', 'Search GPT-5'),
]

summary_data = []

for csv_file, model_name in models_to_compare:
    if not os.path.exists(csv_file):
        print(f"File not found: {csv_file}")
        continue
    
    df = pd.read_csv(csv_file)
    
    total = len(df)
    compiled_count = df['Compiled'].sum()
    correct_count = df['Correctness'].sum()
    
    correct_df = df[df['Correctness'] == True]
    
    # Calculate fast_1 and fast_2 metrics for uncompiled baseline
    fast_1_count = len(df[(df['Correctness'] == True) & (df['Speedup'] > 1.0)])
    fast_2_count = len(df[(df['Correctness'] == True) & (df['Speedup'] >= 2.0)])
    
    # Calculate fast_1 and fast_2 metrics for compiled baseline
    fast_1_comp_count = len(df[(df['Correctness'] == True) & (df['Speedup_vs_Compiled'] > 1.0)])
    fast_2_comp_count = len(df[(df['Correctness'] == True) & (df['Speedup_vs_Compiled'] >= 2.0)])
    
    if len(correct_df) > 0:
        speedup_vals = correct_df['Speedup'].values
        speedup_comp_vals = correct_df['Speedup_vs_Compiled'].values
        
        summary_data.append({
            'Model': model_name,
            'Total': total,
            'Compiled': f"{compiled_count}/{total} ({100*compiled_count/total:.1f}%)",
            'Correct': f"{correct_count}/{total} ({100*correct_count/total:.1f}%)",
            'Fast1/Fast2': f"{fast_1_count}/{fast_2_count}",
            'Speedup': f"{np.mean(speedup_vals):.2f}x ± {np.std(speedup_vals):.2f}",
            'Fast1/Fast2_Comp': f"{fast_1_comp_count}/{fast_2_comp_count}",
            'SpeedupComp': f"{np.mean(speedup_comp_vals):.2f}x ± {np.std(speedup_comp_vals):.2f}"
        })
    else:
        summary_data.append({
            'Model': model_name,
            'Total': total,
            'Compiled': f"{compiled_count}/{total} ({100*compiled_count/total:.1f}%)",
            'Correct': f"{correct_count}/{total} ({100*correct_count/total:.1f}%)",
            'Fast1/Fast2': f"{fast_1_count}/{fast_2_count}",
            'Speedup': "0.00x ± 0.00",
            'Fast1/Fast2_Comp': f"{fast_1_comp_count}/{fast_2_comp_count}",
            'SpeedupComp': "0.00x ± 0.00"
        })

if summary_data:
    results_df = pd.DataFrame(summary_data)
    print("\n" + "="*120)
    print("MODEL PERFORMANCE - GPT-5, Gemini-2.5-Pro, Search GPT-5")
    print("="*120)
    print(results_df.to_string(index=False))
    print("="*120)
else:
    print("No results found. Check if CSV files exist.")



MODEL PERFORMANCE - GPT-5, Gemini-2.5-Pro, Search GPT-5
         Model  Total     Compiled      Correct Fast1/Fast2      Speedup Fast1/Fast2_Comp  SpeedupComp
         GPT-5     10 9/10 (90.0%) 5/10 (50.0%)         3/1 4.04x ± 6.63              1/0 0.84x ± 0.47
Gemini-2.5-Pro     10 3/10 (30.0%) 2/10 (20.0%)         1/1 1.86x ± 1.78              0/0 0.20x ± 0.14
  Search GPT-5     10 7/10 (70.0%) 7/10 (70.0%)         6/2 3.69x ± 5.81              4/1 1.33x ± 0.84


# Comprehensive Model Comparison
## Analysis across all models and levels


In [14]:
import os
import numpy as np
import pandas as pd

def analyze_model_csv(csv_path, model_name):
    """
    Analyze a model's CSV file and return statistics for each level.
    
    Args:
        csv_path: Path to the CSV file
        model_name: Name of the model for display
    
    Returns:
        DataFrame with statistics per level
    """
    if not os.path.exists(csv_path):
        print(f"File not found: {csv_path}")
        return None
    
    df = pd.read_csv(csv_path)
    
    results = []
    
    for level in ['level1', 'level2', 'level3']:
        df_level = df[df['Level'] == level]
        
        if len(df_level) == 0:
            continue
        
        # Count compiled and correctness
        total = len(df_level)
        compiled_count = df_level['Compiled'].sum()
        correct_count = df_level['Correctness'].sum()
        
        # Calculate speedup statistics for correct results only
        correct_df = df_level[df_level['Correctness'] == True]
        
        # Calculate fast_1 and fast_2 metrics for uncompiled baseline
        # fast_1: correct and faster than PyTorch baseline (speedup > 1.0)
        fast_1_count = len(df_level[(df_level['Correctness'] == True) & (df_level['Speedup'] > 1.0)])
        # fast_2: correct and at least 2x faster than PyTorch baseline (speedup >= 2.0)
        fast_2_count = len(df_level[(df_level['Correctness'] == True) & (df_level['Speedup'] >= 2.0)])
        
        # Calculate fast_1 and fast_2 metrics for compiled baseline
        fast_1_comp_count = len(df_level[(df_level['Correctness'] == True) & (df_level['Speedup_vs_Compiled'] > 1.0)])
        fast_2_comp_count = len(df_level[(df_level['Correctness'] == True) & (df_level['Speedup_vs_Compiled'] >= 2.0)])
        
        if len(correct_df) > 0:
            # Speedup (vs uncompiled PyTorch)
            speedup_vals = correct_df['Speedup'].values
            mean_speedup = np.mean(speedup_vals)
            std_speedup = np.std(speedup_vals)
            
            # Speedup vs compiled PyTorch
            speedup_comp_vals = correct_df['Speedup_vs_Compiled'].values
            mean_speedup_compiled = np.mean(speedup_comp_vals)
            std_speedup_compiled = np.std(speedup_comp_vals)
        else:
            mean_speedup = std_speedup = 0.0
            mean_speedup_compiled = std_speedup_compiled = 0.0
        
        results.append({
            'Model': model_name,
            'Level': level.replace('level', 'L'),
            'Total': total,
            'Compiled': f"{compiled_count}/{total} ({100*compiled_count/total:.1f}%)",
            'Correct': f"{correct_count}/{total} ({100*correct_count/total:.1f}%)",
            'Fast1/Fast2': f"{fast_1_count}/{fast_2_count}",
            'Speedup': f"{mean_speedup:.2f}x ± {std_speedup:.2f}",
            'Fast1/Fast2_Comp': f"{fast_1_comp_count}/{fast_2_comp_count}",
            'SpeedupComp': f"{mean_speedup_compiled:.2f}x ± {std_speedup_compiled:.2f}"
        })
    
    return pd.DataFrame(results)

def compare_all_models():
    """
    Compare all available model CSV files and return a comprehensive table.
    """
    models = [
        ('zeroshot_gpt_oss_20b.csv', 'GPT-OSS-20B'),
        ('zeroshot_qwen3_4b_base.csv', 'Qwen3-4B-Base'),
        ('zeroshot_qwen3_4b_finetuned.csv', 'Qwen3-4B-Finetuned'),
        ('zeroshot_qwen3_8b_base.csv', 'Qwen3-8B-Base'),
        ('zeroshot_qwen3_8b_finetuned.csv', 'Qwen3-8B-Finetuned'),
        ('zeroshot_qwen3_30b.csv', 'Qwen3-30B'),
        ('zeroshot_qwen3_235b_base.csv', 'Qwen3-235B'),
    ]
    
    all_results = []
    
    for csv_file, model_name in models:
        result = analyze_model_csv(csv_file, model_name)
        if result is not None:
            all_results.append(result)
    
    if all_results:
        combined_df = pd.concat(all_results, ignore_index=True)
        return combined_df
    else:
        return None


In [15]:
# Run the comprehensive analysis
results_table = compare_all_models()

if results_table is not None:
    print("\n" + "="*120)
    print("COMPREHENSIVE MODEL COMPARISON ACROSS ALL LEVELS")
    print("="*120)
    print(results_table.to_string(index=False))
    print("="*120)
else:
    print("No results found")



COMPREHENSIVE MODEL COMPARISON ACROSS ALL LEVELS
             Model Level  Total       Compiled        Correct Fast1/Fast2        Speedup Fast1/Fast2_Comp    SpeedupComp
       GPT-OSS-20B    L1    100 78/100 (78.0%) 11/100 (11.0%)         7/4  7.43x ± 18.48              6/2  7.06x ± 18.97
       GPT-OSS-20B    L2    100 87/100 (87.0%)   1/100 (1.0%)         0/0   1.00x ± 0.00              0/0   0.81x ± 0.00
       GPT-OSS-20B    L3     50  34/50 (68.0%)    3/50 (6.0%)         1/0   0.99x ± 0.07              0/0   0.74x ± 0.07
     Qwen3-4B-Base    L1    100 55/100 (55.0%) 12/100 (12.0%)         4/1   1.20x ± 0.66              3/0   0.93x ± 0.14
     Qwen3-4B-Base    L2    100 49/100 (49.0%)   3/100 (3.0%)         3/0   1.02x ± 0.01              0/0   0.73x ± 0.06
     Qwen3-4B-Base    L3     50  23/50 (46.0%)   8/50 (16.0%)         3/0   1.01x ± 0.04              1/0   0.83x ± 0.11
Qwen3-4B-Finetuned    L1    100   2/100 (2.0%)   0/100 (0.0%)         0/0   0.00x ± 0.00              0

In [16]:
if results_table is not None:
    results_table.style.set_properties(**{
        'text-align': 'left',
        'font-size': '11px'
    }).set_table_styles([
        {'selector': 'th', 'props': [('font-weight', 'bold'), ('background-color', '#f0f0f0')]}
    ])


## Summary Statistics by Model


In [17]:
# Calculate overall statistics per model
if results_table is not None:
    # Group by model and calculate aggregate statistics
    models = [
        ('zeroshot_gpt_oss_20b.csv', 'GPT-OSS-20B'),
        ('zeroshot_qwen3_4b_base.csv', 'Qwen3-4B-Base'),
        ('zeroshot_qwen3_4b_finetuned.csv', 'Qwen3-4B-Finetuned'),
        ('zeroshot_qwen3_8b_base.csv', 'Qwen3-8B-Base'),
        ('zeroshot_qwen3_8b_finetuned.csv', 'Qwen3-8B-Finetuned'),
        ('zeroshot_qwen3_30b.csv', 'Qwen3-30B'),
        ('zeroshot_qwen3_235b_base.csv', 'Qwen3-235B'),
    ]
    
    summary_data = []
    
    for csv_file, model_name in models:
        if not os.path.exists(csv_file):
            continue
        
        df = pd.read_csv(csv_file)
        
        total = len(df)
        compiled_count = df['Compiled'].sum()
        correct_count = df['Correctness'].sum()
        
        correct_df = df[df['Correctness'] == True]
        
        # Calculate fast_1 and fast_2 metrics for uncompiled baseline
        # fast_1: correct and faster than PyTorch baseline (speedup > 1.0)
        fast_1_count = len(df[(df['Correctness'] == True) & (df['Speedup'] > 1.0)])
        # fast_2: correct and at least 2x faster than PyTorch baseline (speedup >= 2.0)
        fast_2_count = len(df[(df['Correctness'] == True) & (df['Speedup'] >= 2.0)])
        
        # Calculate fast_1 and fast_2 metrics for compiled baseline
        fast_1_comp_count = len(df[(df['Correctness'] == True) & (df['Speedup_vs_Compiled'] > 1.0)])
        fast_2_comp_count = len(df[(df['Correctness'] == True) & (df['Speedup_vs_Compiled'] >= 2.0)])
        
        if len(correct_df) > 0:
            speedup_vals = correct_df['Speedup'].values
            speedup_comp_vals = correct_df['Speedup_vs_Compiled'].values
            
            summary_data.append({
                'Model': model_name,
                'Total': total,
                'Compiled_Rate': f"{100*compiled_count/total:.1f}%",
                'Correct_Rate': f"{100*correct_count/total:.1f}%",
                'Fast1/Fast2': f"{fast_1_count}/{fast_2_count}",
                'Speedup': f"{np.mean(speedup_vals):.2f}x ± {np.std(speedup_vals):.2f}",
                'Fast1/Fast2_Comp': f"{fast_1_comp_count}/{fast_2_comp_count}",
                'SpeedupComp': f"{np.mean(speedup_comp_vals):.2f}x ± {np.std(speedup_comp_vals):.2f}"
            })
        else:
            summary_data.append({
                'Model': model_name,
                'Total': total,
                'Compiled_Rate': f"{100*compiled_count/total:.1f}%",
                'Correct_Rate': f"{100*correct_count/total:.1f}%",
                'Fast1/Fast2': f"{fast_1_count}/{fast_2_count}",
                'Speedup': "0.00x ± 0.00",
                'Fast1/Fast2_Comp': f"{fast_1_comp_count}/{fast_2_comp_count}",
                'SpeedupComp': "0.00x ± 0.00"
            })
    
    summary_df = pd.DataFrame(summary_data)
    print("\n" + "="*100)
    print("OVERALL MODEL PERFORMANCE SUMMARY (Across All Levels)")
    print("="*100)
    print(summary_df.to_string(index=False))
    print("="*100)



OVERALL MODEL PERFORMANCE SUMMARY (Across All Levels)
             Model  Total Compiled_Rate Correct_Rate Fast1/Fast2       Speedup Fast1/Fast2_Comp   SpeedupComp
       GPT-OSS-20B    250         79.6%         6.0%         8/4 5.72x ± 16.08              6/2 5.38x ± 16.48
     Qwen3-4B-Base    250         50.8%         9.2%        10/1  1.11x ± 0.49              4/0  0.87x ± 0.14
Qwen3-4B-Finetuned    250          6.4%         0.0%         0/0  0.00x ± 0.00              0/0  0.00x ± 0.00
     Qwen3-8B-Base    250         82.8%         7.6%         8/1  1.04x ± 0.73              8/0  0.77x ± 0.44
Qwen3-8B-Finetuned    250          8.0%         0.8%         1/0  1.05x ± 0.07              0/0  0.78x ± 0.20
         Qwen3-30B    250         70.0%         3.2%         4/2 9.10x ± 19.82              5/3 9.34x ± 20.22
        Qwen3-235B    250         92.8%        12.8%       13/10  3.09x ± 9.20             18/6  2.86x ± 9.29
